# 数据整理：连接、合并和重塑

在许多应用中，数据可能分布在多个文件或数据库中，或者以不方便分析的形式排列。本章重点介绍帮助合并、连接和重新排列数据的工具。

首先，我介绍了pandas中层次索引的概念，这在一些操作中被广泛使用。然后，我深入探讨了特定的数据操作。你可以在第13章：数据分析示例中看到这些工具的各种应用。

## 分层索引

层次索引是pandas的一个重要特性，它允许你在一个轴上拥有多个（两个或更多）索引级别。另一种思考方式是，它提供了一种以低维形式处理高维数据的方法。让我们从一个简单的例子开始：创建一个Series，其索引是一个列表的列表（或数组）：

In [2]:
import pandas as pd
import numpy as np

In [3]:
data = pd.Series(np.random.uniform(size=9),
                 index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                        [1, 2, 3, 1, 2, 3, 1, 2, 3]])
data

a  1    0.302384
   2    0.896260
   3    0.839349
b  1    0.253476
   2    0.425330
c  3    0.354937
   1    0.486085
d  2    0.736453
   3    0.267726
dtype: float64

您看到的是一张具有多重索引的序列的美化视图。索引显示中的“间隙”意味着“直接使用上面的标签”：

In [4]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 2),
            ('c', 3),
            ('c', 1),
            ('d', 2),
            ('d', 3)],
           )

对于层次索引的对象，可以进行所谓的部分索引，从而可以简洁地选择数据的子集：

In [5]:
data['b']

1    0.253476
2    0.425330
dtype: float64

In [6]:
data['b':'c']

b  1    0.253476
   2    0.425330
c  3    0.354937
   1    0.486085
dtype: float64

In [7]:
data.loc[["b", "d"], [1, 2]]

b  1    0.253476
   2    0.425330
d  2    0.736453
dtype: float64

甚至可以从“内部”级别进行选择。在这里，我从第二个索引级别选择了所有值为2的值：

In [8]:
data.loc[:, 2]

a    0.896260
b    0.425330
d    0.736453
dtype: float64

层次索引在重塑数据和基于组的操作（如创建透视表）中扮演着重要角色。例如，您可以使用其unstack方法将此数据重新排列为DataFrame：

In [9]:
data.unstack()

,1,2,3
a,0.302384,0.896260,0.839349
b,0.253476,0.425330,NaN
c,0.486085,NaN,0.354937
d,NaN,0.736453,0.267726


unstack的逆操作是stack：

In [10]:
data.unstack().stack()

a  1    0.302384
   2    0.896260
   3    0.839349
b  1    0.253476
   2    0.425330
c  1    0.486085
   3    0.354937
d  2    0.736453
   3    0.267726
dtype: float64

DataFrame的任一轴都可以有一个层次索引：

In [11]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[['a', 'a', 'b', 'b'],
                            [1, 2, 1, 2]],
                     columns=[['Ohio', 'Ohio', 'Colorado'],
                              ['Green', 'Red', 'Green']])
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

层次结构级别可以具有名称（作为字符串或任何Python对象）。如果是这样，这些将在控制台输出中显示：

In [12]:
frame.index.names = ['key1', 'key2']
frame.columns.names = ['state', 'color']
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

names替代了仅用于单层索引的name属性。
> 请注意，“state”和“color”索引名称不是行标签（frame.index值）的一部分。

通过访问其nlevels属性，你可以看到索引有多少个级别：

In [13]:
frame.index.nlevels

2

使用部分列索引，您可以类似地选择一组列：

In [14]:
frame['Ohio']

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

MultiIndex可以单独创建然后重用；前面的DataFrame中的列也可以像这样创建：

In [15]:
pd.MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'],
                           ['Green', 'Red', 'Green']],
                          names=['state', 'color'])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

### 重排序和排序级别

有时您可能需要重新排列轴上级别的顺序或按某个特定级别中的值对数据进行排序。swaplevel方法接受两个级别编号或名称，并返回一个新对象，其中级别的顺序已交换（但数据未更改）：

In [16]:
frame.swaplevel('key1', 'key2')

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

sort_index 默认使用所有索引级别按字典顺序对数据进行排序，但您可以通过传递 level 参数来选择仅使用单个级别或子集进行排序。例如：

In [17]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [18]:
frame.swaplevel(0, 1).sort_index(level=0)

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

> 如果索引从最外层开始按字典顺序排序，那么在层次索引对象上的数据选择性能会更好——也就是说，调用sort_index(level=0)或sort_index()的结果。

### 按级别汇总统计

DataFrame和Series上的许多描述性和摘要统计都有一个level选项，您可以在特定轴上指定要聚合的级别。考虑上述DataFrame；我们可以通过行或列对级别进行聚合，如下所示：

In [19]:
frame.groupby(level='key2').sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

In [20]:
frame.groupby(level='color', axis="columns").sum()

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_27661/141794815.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  frame.groupby(level='color', axis="columns").sum()


color      Green  Red
key1 key2            
a    1         2    1
     2         8    4
b    1        14    7
     2        20   10

### 使用DataFrame的列进行索引

使用DataFrame的一列或多列作为行索引并不罕见；或者，你可能希望将行索引移动到DataFrame的列中。这里有一个示例DataFrame：

In [21]:
frame = pd.DataFrame({"a": range(7),
                      "b": range(7, 0, -1),
                      "c": ['one', 'one', 'one', 'two', 'two', 'two', 'two'],
                      "d": [0, 1, 2, 0, 1, 2, 3]})
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


DataFrame的set_index函数将使用其一个或多个列作为索引来创建一个新的DataFrame：

In [22]:
frame2 = frame.set_index(['c', 'd'])
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

默认情况下，列将从DataFrame中删除。不过，您可以通过传递drop=False给set_index来保留它们：

In [23]:
frame.set_index(['c', 'd'], drop=False)

a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

另一方面，reset_index与set_index相反；层次索引级别被移动到列中：

In [24]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


## 数据集的结合与合并

pandas对象中包含的数据可以通过多种方式进行组合：

**pandas.merge**
根据一个或多个键将DataFrame中的行连接起来。这对SQL或其他关系数据库的用户来说很熟悉，因为它实现了数据库的连接操作。

**pandas.concat**
沿一个轴将对象连接或“堆叠”在一起。

**combine_first**
将重叠的数据拼接在一起，用另一个对象中的值填充一个对象中缺失的值。

### 数据库风格的DataFrame连接

合并或连接操作通过使用一个或多个键来链接行，从而将数据集组合在一起。这些操作在关系数据库中特别重要（例如，基于SQL的）。pandas中的pandas.merge函数是使用这些算法处理数据的主要入口点。

让我们从一个简单的例子开始：

In [25]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],
                    'data2': range(3)})

In [26]:
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [27]:
df2

,key,data2
0,a,0
1,b,1
2,d,2


这是一个多对一连接的例子；df1中的数据有多行标记为a和b，而df2中每个键列的值只有一行。调用pandas.merge与这些对象一起，我们得到：

In [28]:
pd.merge(df1, df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


请注意，我没有指定要连接哪一列。如果未指定此信息，pandas.merge将使用重叠的列名作为键。尽管如此，明确指定是一个好习惯：

In [29]:
pd.merge(df1, df2, on='key')

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


通常情况下，pandas.merge操作中列输出的顺序是不确定的。

如果每个对象中的列名不同，您可以分别指定它们：

In [30]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'],
                    'data2': range(3)})
pd.merge(df3, df4, left_on='lkey', right_on='rkey')

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,a,2,a,0
3,a,4,a,0
4,a,5,a,0
5,b,6,b,1


您可能会注意到结果中缺少了“c”和“d”值及相关数据。默认情况下，pandas.merge执行的是“内连接”；结果中的键是交集，或者在两个表中都找到的共同集合。其他可能的选项有“左连接”、“右连接”和“外连接”。外连接取键的并集，结合了应用左连接和右连接的效果：

In [31]:
pd.merge(df1, df2, how='outer')

,key,data1,data2
0,a,2.0,0.0
1,a,4.0,0.0
2,a,5.0,0.0
3,b,0.0,1.0
4,b,1.0,1.0
5,b,6.0,1.0
6,c,3.0,NaN
7,d,NaN,2.0


In [32]:
pd.merge(df3, df4, left_on='lkey', right_on='rkey', how='outer')

,lkey,data1,rkey,data2
0,a,2.0,a,0.0
1,a,4.0,a,0.0
2,a,5.0,a,0.0
3,b,0.0,b,1.0
4,b,1.0,b,1.0
5,b,6.0,b,1.0
6,c,3.0,NaN,NaN
7,NaN,NaN,d,2.0


在外连接中，如果左表或右表的行在另一个数据框的键上没有匹配项，则这些行将在另一个数据框的非匹配行的列中出现NA值。

**参数how的不同连接类型:**
| 连接类型 | 行为 |
|---------|------|
| how="inner" | 只使用两个表中都存在的键组合 |
| how="left" | 使用左表中列出的所有键组合 |
| how="right" | 使用右表中找到的所有键组合 |
| how="outer" | 使用两个表中观察到的所有键组合 |

多对多的合并形成了匹配键的笛卡尔积。这里有一个例子：

In [33]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],
                    'data1': range(6)})
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'],
                    'data2': range(5)})
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [34]:
df2

,key,data2
0,a,0
1,b,1
2,a,2
3,b,3
4,d,4


In [35]:
pd.merge(df1, df2, on='key', how='left')

,key,data1,data2
0,b,0,1.0
1,b,0,3.0
2,b,1,1.0
3,b,1,3.0
4,a,2,0.0
5,a,2,2.0
6,c,3,NaN
7,a,4,0.0
8,a,4,2.0
9,b,5,1.0


由于左数据框中有三个“b”行，右数据框中有两个“b”行，因此结果中有六个“b”行。传递给how关键字参数的join方法只影响结果中出现的不同键值：

In [36]:
pd.merge(df1, df2, how='inner')

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,a,4,0
7,a,4,2
8,b,5,1
9,b,5,3


要合并多个键，请传递一个列名列表：

In [37]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],
                     'key2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],
                      'key2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})
pd.merge(left, right, on=['key1', 'key2'], how='outer')

,key1,key2,lval,rval
0,bar,one,3.0,6.0
1,bar,two,NaN,7.0
2,foo,one,1.0,4.0
3,foo,one,1.0,5.0
4,foo,two,2.0,NaN


要确定根据合并方法选择的结果中会出现哪些键组合，请考虑多个键作为元组数组形成单个连接键。
> 当你在多个列上连接时，传递的DataFrame对象上的索引将被丢弃。如果你需要保留索引值，可以使用reset_index来将索引附加到列中。

在合并操作中需要考虑的最后一个问题是重叠列名的处理。例如：

In [38]:
pd.merge(left, right, on='key1')

,key1,key2_x,lval,key2_y,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


虽然您可以手动解决重叠问题（请参见第7.2.4节：重命名轴索引以重命名轴标签），但pandas.merge有一个suffixes选项，用于指定要附加到左右DataFrame对象中重叠名称的字符串：

In [39]:
pd.merge(left, right, on='key1', suffixes=('_left', '_right'))

,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


**pandas.merge函数参数:**
| 参数 | 描述 |
|------|-----|
| left | 要在左侧合并的 DataFrame |
| right | 要在右侧合并的 DataFrame |
| how | 要应用的连接类型："inner"、"outer"、"left"或"right"之一；默认为"inner" |
| on | 要连接的列名。必须在两个DataFrame对象中都找到。如果没有指定并且没有给出其他连接键，将使用左右两边的列名的交集作为连接键 |
| left_on | 左数据框中用作连接键的列。可以是单个列名或列名列表 |
| right_on | 类似于right DataFrame的left_on |
| left_index | 使用左边的行索引作为其连接键（如果是一个MultiIndex，则使用多个键）|
| right_index | 类似于left_index |
| sort | 按连接键对合并数据进行字典序排序；默认值为False |
| suffixes | 在列名重叠的情况下要附加到列名中的字符串值元组；默认值为（“_x”，“_y”）（例如，如果两个DataFrame对象中都有“data”，则结果中将显示为“data_x”和“data_y”）|
| copy | 如果为False，在某些特殊情况下避免将数据复制到结果数据结构中；默认总是复制 |
| validate | 验证合并是否为指定类型，是一对一、一对多或多对多。有关选项的完整详细信息，请参见文档字符串。|
| indicator | 添加一个特殊列_merge，该列指示每行的来源；值将根据每行中连接数据的来源是“仅左侧”、“仅右侧”还是“两侧”而有所不同 |

### 索引合并

在某些情况下，DataFrame中的合并键可能位于其索引（行标签）中。在这种情况下，您可以传递left_index=True或right_index=True（或两者都传递）来指示应使用索引作为合并键：

In [40]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],
                      'value': range(6)})
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])

left1

,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


In [41]:
right1

,group_val
a,3.5
b,7.0


In [42]:
pd.merge(left1, right1, left_on='key', right_index=True)

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0


> 如果您仔细观察这里，您会看到left1的索引值已被保留，而在上面的其他示例中，输入DataFrame对象的索引被丢弃。因为right1的索引是唯一的，这种“多对一”合并（默认方法为how="inner"）可以保留left1中与输出中的行相对应的索引值。

由于默认的合并方法是相交连接键，因此您可以使用外连接来形成它们的并集：

In [43]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer')

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0
5,c,5,NaN


对于层次索引数据来说，情况要复杂得多，因为按索引连接相当于多键合并：

In [44]:
lefth = pd.DataFrame({'key1': ['Ohio', 'Ohio', 'Ohio', 'Nevada', 'Nevada'],
                      'key2': [2000, 2001, 2002, 2001, 2002],
                      'data': np.arange(5.)})
righth_index = pd.MultiIndex.from_arrays(
    [
        ["Nevada", "Nevada", "Ohio", "Ohio", "Ohio", "Ohio"],
        [2001, 2000, 2000, 2000, 2001, 2002],
    ]
)

righth = pd.DataFrame({"event1": pd.Series([0, 2, 4, 6, 8, 10], index=righth_index), 
                       "event2": pd.Series([1, 3, 5, 7, 9, 11], index=righth_index)})

lefth

,key1,key2,data
0,Ohio,2000,0.0
1,Ohio,2001,1.0
2,Ohio,2002,2.0
3,Nevada,2001,3.0
4,Nevada,2002,4.0


In [45]:
righth

event1  event2
Nevada 2001       0       1
       2000       2       3
Ohio   2000       4       5
       2000       6       7
       2001       8       9
       2002      10      11

在这种情况下，您必须将要合并的多个列作为列表指示（注意如何处理重复的索引值，使用how="outer"）：

In [46]:
pd.merge(lefth, righth, left_on=['key1', 'key2'], right_index=True)

,key1,key2,data,event1,event2
0,Ohio,2000,0.0,4,5
0,Ohio,2000,0.0,6,7
1,Ohio,2001,1.0,8,9
2,Ohio,2002,2.0,10,11
3,Nevada,2001,3.0,0,1


In [47]:
pd.merge(lefth, righth, left_on=['key1', 'key2'], right_index=True, how='outer')

,key1,key2,data,event1,event2
4,Nevada,2000,NaN,2.0,3.0
3,Nevada,2001,3.0,0.0,1.0
4,Nevada,2002,4.0,NaN,NaN
0,Ohio,2000,0.0,4.0,5.0
0,Ohio,2000,0.0,6.0,7.0
1,Ohio,2001,1.0,8.0,9.0
2,Ohio,2002,2.0,10.0,11.0


也可以使用合并两边的索引：

In [48]:
left2 = pd.DataFrame([[1, 2], [3, 4], [5, 6]], index=['a', 'c', 'e'], columns=['Ohio', 'Nevada'])
right2 = pd.DataFrame([[7, 8], [9, 10], [11, 12], [13, 14]], index=['b', 'c', 'd', 'e'], columns=['Missouri', 'Alabama'])

left2

,Ohio,Nevada
a,1,2
c,3,4
e,5,6


In [49]:
right2

,Missouri,Alabama
b,7,8
c,9,10
d,11,12
e,13,14


In [50]:
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)

,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


DataFrame有一个join实例方法来简化通过索引合并。它也可以用来组合许多具有相同或相似索引但列不重叠的DataFrame对象。在之前的例子中，我们可以这样写：

In [51]:
left2.join(right2, how='outer')

,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


与pandas.merge相比，DataFrame的join方法默认执行左连接。它还支持将传入DataFrame的索引与调用DataFrame的一列进行连接：

In [52]:
left1.join(right1, on='key')

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0
5,c,5,NaN


你可以将这种方法视为将数据“合并”到调用其join方法的对象中。最后，对于简单的索引对索引的合并，你可以传递一个DataFrame列表作为参数来执行合并，而不是使用下一节描述的更通用的pandas.concat函数：

In [53]:
another = pd.DataFrame([[7, 8], [9, 10], [11, 12], [16, 17]],
                       index=['a', 'c', 'e', 'f'], 
                       columns=['New York', 'Oregon'])
another

,New York,Oregon
a,7,8
c,9,10
e,11,12
f,16,17


In [54]:
left2.join([right2, another])

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0


In [55]:
left2.join([right2, another], how='outer')

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0
b,NaN,NaN,7.0,8.0,NaN,NaN
d,NaN,NaN,11.0,12.0,NaN,NaN
f,NaN,NaN,NaN,NaN,16.0,17.0


### 沿轴连接

另一种数据组合操作可以互换地称为连接或堆叠。NumPy的concatenate函数可以对NumPy数组执行此操作：

In [56]:
arr = np.arange(12).reshape((3, 4))
arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [57]:
np.concatenate([arr, arr], axis=1)

array([[ 0,  1,  2,  3,  0,  1,  2,  3],
       [ 4,  5,  6,  7,  4,  5,  6,  7],
       [ 8,  9, 10, 11,  8,  9, 10, 11]])

在pandas对象（如Series和DataFrame）的背景下，带有标签的轴使您可以进一步推广数组连接。特别是，您有许多额外的考虑因素：
- 如果这些对象在其他轴上的索引不同，我们应该合并这些轴上不同的元素还是只使用共同的值？
- 在生成的对象中，这些连接的数据块需要被识别出来吗？
- “连接轴”是否包含需要保留的数据？在许多情况下，DataFrame中的默认整数标签在连接过程中最好被丢弃。

pandas中的concat函数提供了一种一致的方法来解决这些问题。我将给出一些例子来说明它是如何工作的。假设我们有三个没有索引重叠的Series：

In [58]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])
s3 = pd.Series([5, 6], index=['f', 'g'])

将这些对象放入列表中调用pandas.concat会将值和索引粘合在一起：

In [59]:
s1

a    0
b    1
dtype: int64

In [60]:
s2

c    2
d    3
e    4
dtype: int64

In [61]:
s3

f    5
g    6
dtype: int64

In [62]:
pd.concat([s1, s2, s3])

a    0
b    1
c    2
d    3
e    4
f    5
g    6
dtype: int64

默认情况下，pandas.concat沿轴“索引”工作，产生另一个Series。如果您传递轴“列”，结果将改为DataFrame：

In [63]:
pd.concat([s1, s2, s3], axis='columns')

,0,1,2
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


在这种情况下，另一个轴上没有重叠，正如你所看到的，这是索引的并集（"外部"连接）。你可以通过传递join="inner"来交叉它们：

In [64]:
s4 = pd.concat([s1, s3])
s4

a    0
b    1
f    5
g    6
dtype: int64

In [65]:
pd.concat([s1, s4], axis='columns')

,0,1
a,0.0,0
b,1.0,1
f,NaN,5
g,NaN,6


In [66]:
pd.concat([s1, s4], axis='columns', join='inner')

,0,1
a,0,0
b,1,1


在最后一个例子中，“f”和“g”标签消失了，因为使用了join="inner"选项。

潜在的问题是，在结果中无法识别连接的片段。假设相反，您希望在连接轴上创建一个层次索引。为此，请使用keys参数：

In [67]:
result = pd.concat([s1, s1, s3], keys=['one', 'two', 'three'])
result

one    a    0
       b    1
two    a    0
       b    1
three  f    5
       g    6
dtype: int64

In [68]:
result.unstack()

,a,b,f,g
one,0.0,1.0,NaN,NaN
two,0.0,1.0,NaN,NaN
three,NaN,NaN,5.0,6.0


在axis="columns"组合Series的情况下，keys将变成DataFrame的列标题：

In [69]:
pd.concat([s1, s2, s3], axis='columns', keys=['one', 'two', 'three'])

,one,two,three
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


同样的逻辑也适用于DataFrame对象：

In [70]:
df1 = pd.DataFrame(np.arange(6).reshape((3, 2)),
                   index=['a', 'b', 'c'],
                   columns=['one', 'two'])
df2 = pd.DataFrame(5 + np.arange(4).reshape((2, 2)),
                   index=['a', 'c'],
                   columns=['three', 'four'])
df1

,one,two
a,0,1
b,2,3
c,4,5


In [71]:
df2

,three,four
a,5,6
c,7,8


In [72]:
pd.concat([df1, df2], axis='columns', keys=['level1', 'level2'])

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

这里使用keys参数创建了一个层次索引，其中第一层可以用来识别每个连接的DataFrame对象。

如果您传递一个对象字典而不是列表，则字典的键将用作键选项：

In [73]:
pd.concat({'level1': df1, 'level2': df2}, axis='columns')

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

创建层次索引的附加参数（见表 8.3）。例如，我们可以使用名称参数来命名创建的轴级别：

In [74]:
pd.concat([df1, df2], axis='columns', keys=['level1', 'level2'], names=['upper', 'lower'])

upper level1     level2     
lower    one two  three four
a          0   1    5.0  6.0
b          2   3    NaN  NaN
c          4   5    7.0  8.0

最后一个考虑因素涉及行索引不包含任何相关数据的DataFrame：

In [75]:
df1 = pd.DataFrame(np.random.randn(3, 4), columns=['a', 'b', 'c', 'd'])
df2 = pd.DataFrame(np.random.randn(2, 3), columns=['b', 'd', 'a'])
df1

,a,b,c,d
0,1.043221,0.212029,0.284961,-0.314389
1,-0.471793,0.282106,0.041569,0.424473
2,-1.238870,1.941667,-1.724816,-0.305276


In [76]:
df2

,b,d,a
0,-0.627094,0.185800,-1.041076
1,0.875167,-0.465254,-1.618570


在这种情况下，您可以传递ignore_index=True，这将丢弃每个DataFrame中的索引，并仅将列中的数据连接起来，分配一个新的默认索引：

In [77]:
pd.concat([df1, df2], ignore_index=True)

,a,b,c,d
0,1.043221,0.212029,0.284961,-0.314389
1,-0.471793,0.282106,0.041569,0.424473
2,-1.238870,1.941667,-1.724816,-0.305276
3,-1.041076,-0.627094,NaN,0.185800
4,-1.618570,0.875167,NaN,-0.465254


**pandas.concat函数参数:**
| 参数 | 描述 |
|-----|------|
| objs | 要连接的pandas对象的列表或字典；这是唯一必需的参数 |
| axis | 要沿其连接的轴；默认情况下是沿行（axis="index"）连接 |
| join | 内部或外部（默认为外部）；沿其他轴是相交（内部）还是并集（外部）|
| keys | 要关联的对象值，形成沿连接轴的层次索引；可以是任意值的列表或数组、元组数组或数组列表（如果传入的是多级数组）|
| levels | 如果传递了键，则作为层次索引级别或级别的特定索引 |
| names | 如果传递了键和/或层次结构，则创建的层次结构的名称 |
| verify_integrity | 检查连接对象中的新轴是否有重复项，如果有则抛出异常；默认情况下（False）允许重复 |
| ignore_index | 不要在连接轴上保留索引，而是生成一个新的range(total_length)索引 |

### 结合重叠数据

还有一种数据组合情况既不能表示为合并操作也不能表示为连接操作。你可能有两个数据集，它们的索引完全或部分重叠。作为一个激励示例，考虑NumPy的where函数，它执行的是数组导向的if-else表达式的等效操作：

In [78]:
a = pd.Series([np.nan, 2.5, 0.0, 3.5, 4.5, np.nan], 
              index=['f', 'e', 'd', 'c', 'b', 'a'])
b = pd.Series([0, np.nan, 2, np.nan, np.nan, 5], 
              index=['a', 'b', 'c', 'd', 'e', 'f'])
a

f    NaN
e    2.5
d    0.0
c    3.5
b    4.5
a    NaN
dtype: float64

In [79]:
b

a    0.0
b    NaN
c    2.0
d    NaN
e    NaN
f    5.0
dtype: float64

In [80]:
np.where(pd.isnull(a), b, a)

array([0. , 2.5, 0. , 3.5, 4.5, 5. ])

在这里，每当a中的值为空时，就会选择b中的值，否则会选择a中非空的值。使用numpy.where不会检查索引标签是否对齐（甚至不需要对象长度相同），所以如果你想按索引对齐值，请使用Series的combine_first方法：

In [81]:
a.combine_first(b)

a    0.0
b    4.5
c    3.5
d    0.0
e    2.5
f    5.0
dtype: float64

在DataFrame中，combine_first逐列执行相同操作，因此可以将其视为用传递对象中的数据“修补”调用对象中的缺失数据：

In [82]:
df1 = pd.DataFrame({'a': [1, np.nan, 5, np.nan],
                    'b': [np.nan, 2, np.nan, 6],
                    'c': range(2, 18, 4)})
df2 = pd.DataFrame({'a': [5, 4, np.nan, 3, 7],
                    'b': [np.nan, 3, 4, 6, 8]})
df1

,a,b,c
0,1.0,NaN,2
1,NaN,2.0,6
2,5.0,NaN,10
3,NaN,6.0,14


In [83]:
df2

,a,b
0,5.0,NaN
1,4.0,3.0
2,NaN,4.0
3,3.0,6.0
4,7.0,8.0


In [84]:
df1.combine_first(df2)

,a,b,c
0,1.0,NaN,2.0
1,4.0,2.0,6.0
2,5.0,4.0,10.0
3,3.0,6.0,14.0
4,7.0,8.0,NaN


使用DataFrame对象调用combine_first时，其输出将包含所有列名的并集。

## 重塑和转置

有许多基本的操作用于重新排列表格数据。这些被称为重塑或转置操作。

### 使用分层索引进行重塑

层次化索引提供了一种一致的方式来重新排列DataFrame中的数据。主要有两个操作：

**stack** 这种“旋转”或枢轴转动是从数据的列到行。
**unstack** 这从行变成了列。

我将通过一系列示例来说明这些操作。考虑一个小型DataFrame，其行和列索引都是字符串数组：

In [85]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(['Ohio', 'Colorado'], name='state'),
                    columns=pd.Index(['one', 'two', 'three'], name='number'))
data

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


使用堆栈方法在这组数据上会将列转换为行，产生一个Series：

In [86]:
result = data.stack()
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int64

从层次索引的Series中，你可以使用unstack将数据重新排列回DataFrame：

In [87]:
result.unstack()

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


默认情况下，最内层是不堆叠的（与堆叠相同）。您可以通过传递一个层级编号或名称来取消堆叠不同的层级：

In [88]:
result.unstack(level=0)

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


In [89]:
result.unstack(level='state')

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


如果该层中的所有值在每个子组中都没有找到，则拆分可能会引入缺失数据：

In [90]:
s1 = pd.Series([0, 1, 2, 3], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([4, 5, 6], index=['c', 'd', 'e'])

data2 = pd.concat([s1, s2], keys=['one', 'two'])
data2

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: int64

堆叠默认会过滤掉缺失数据，因此操作更容易被反演：

In [91]:
data2.unstack()

,a,b,c,d,e
one,0.0,1.0,2.0,3.0,NaN
two,NaN,NaN,4.0,5.0,6.0


In [92]:
data2.unstack().stack()

one  a    0.0
     b    1.0
     c    2.0
     d    3.0
two  c    4.0
     d    5.0
     e    6.0
dtype: float64

In [93]:
data2.unstack().stack(dropna=False)

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_27661/3936770077.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data2.unstack().stack(dropna=False)


one  a    0.0
     b    1.0
     c    2.0
     d    3.0
     e    NaN
two  a    NaN
     b    NaN
     c    4.0
     d    5.0
     e    6.0
dtype: float64

在DataFrame中展开时，展开的层次成为结果中的最低层：

In [94]:
df = pd.DataFrame({'left': result, 
                   'right': result + 5},
                  columns=pd.Index(['left', 'right'], name='side'))
df

side             left  right
state    number             
Ohio     one        0      5
         two        1      6
         three      2      7
Colorado one        3      8
         two        4      9
         three      5     10

In [95]:
df.unstack(level='state')

side   left          right         
state  Ohio Colorado  Ohio Colorado
number                             
one       0        3     5        8
two       1        4     6        9
three     2        5     7       10

与unstack一样，在调用stack时我们可以指定要堆叠的轴的名称：

In [96]:
df.unstack(level='state').stack(level='side')

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_27661/1718662294.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.unstack(level='state').stack(level='side')


state         Ohio  Colorado
number side                 
one    left      0         3
       right     5         8
two    left      1         4
       right     6         9
three  left      2         5
       right     7        10

### 将“长”格式转为“宽”格式

在数据库和CSV文件中存储多个时间序列的常见方式有时被称为长格式或堆叠格式。在这种格式中，单个值由表中的一行表示，而不是每行多个值。

让我们加载一些示例数据，并进行少量的时序数据处理和其他数据清洗：

In [97]:
data = pd.read_csv('examples/macrodata.csv')
data.head()

,year,quarter,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint
0,1959,1,2710.349,1707.4,286.898,470.045,1886.9,28.98,139.7,2.82,5.8,177.146,0.00,0.00
1,1959,2,2778.801,1733.7,310.859,481.301,1919.7,29.15,141.7,3.08,5.1,177.830,2.34,0.74
2,1959,3,2775.488,1751.8,289.226,491.260,1916.4,29.35,140.5,3.82,5.3,178.657,2.74,1.09
3,1959,4,2785.204,1753.7,299.356,484.052,1931.3,29.37,140.0,4.33,5.6,179.386,0.27,4.06
4,1960,1,2847.699,1770.5,331.722,462.199,1955.5,29.54,139.6,3.50,5.2,180.007,2.31,1.19


In [98]:
data = data.loc[:, ['year', 'quarter', 'realgdp', 'infl', 'unemp']]
data.head()

,year,quarter,realgdp,infl,unemp
0,1959,1,2710.349,0.00,5.8
1,1959,2,2778.801,2.34,5.1
2,1959,3,2775.488,2.74,5.3
3,1959,4,2785.204,0.27,5.6
4,1960,1,2847.699,2.31,5.2


首先，我使用pandas.PeriodIndex（它代表时间间隔而不是时间点），在更详细的第11章：时间序列中讨论，将年份和季度列结合起来，设置索引由每个季度末的datetime值组成：

In [99]:
periods = pd.PeriodIndex.from_fields(
    year=data["year"], quarter=data["quarter"]
)
periods.name = 'date'
periods

PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

In [100]:
data.index = periods.to_timestamp('D')
data.head()

,year,quarter,realgdp,infl,unemp
date,,,,,
1959-01-01,1959,1,2710.349,0.00,5.8
1959-04-01,1959,2,2778.801,2.34,5.1
1959-07-01,1959,3,2775.488,2.74,5.3
1959-10-01,1959,4,2785.204,0.27,5.6
1960-01-01,1960,1,2847.699,2.31,5.2


然后，我选择了一列子集，并将这些列的索引命名为“item”：

In [101]:
data = data.reindex(columns=['realgdp', 'infl', 'unemp'])
data.columns.name = 'item'
data.head()

item,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


最后，我用stack重塑数据，用reset_index将新的索引级别转换为列，最后给包含数据值的列命名为“value”：

In [102]:
long_data = data.stack().reset_index().rename(columns={0: 'value'})
long_data.head()

,date,item,value
0,1959-01-01,realgdp,2710.349
1,1959-01-01,infl,0.000
2,1959-01-01,unemp,5.800
3,1959-04-01,realgdp,2778.801
4,1959-04-01,infl,2.340


在所谓的多时间序列长格式中，表格中的每一行代表一个单独的观测值。

数据经常以这种方式存储在关系型SQL数据库中，因为固定模式（列名和数据类型）允许随着向表中添加数据而改变项目列中的不同值的数量。在前面的例子中，日期和项目通常是主键（在关系型数据库术语中），提供了关系完整性和更简单的连接。在某些情况下，这种格式的数据可能更难以处理；您可能希望有一个DataFrame，其中每个不同的项目值都有一个列，按日期列中的时间戳索引。DataFrame的pivot方法执行的就是这种转换：

In [103]:
pivoted = long_data.pivot(index='date', columns='item', values='value')
pivoted.head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


传递的前两个值将分别用作行和列索引，最后是一个可选的值列来填充DataFrame。假设您有两个值列，您希望同时重塑它们：

In [104]:
long_data['value2'] = np.random.standard_normal(len(long_data))
long_data.head()

,date,item,value,value2
0,1959-01-01,realgdp,2710.349,-0.967162
1,1959-01-01,infl,0.000,2.322029
2,1959-01-01,unemp,5.800,1.219199
3,1959-04-01,realgdp,2778.801,-0.137503
4,1959-04-01,infl,2.340,-0.744763


省略最后一个参数后，你将得到一个具有层次结构列的 DataFrame:

In [105]:
pivoted = long_data.pivot(index='date', columns='item')
pivoted.head()

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8  2.322029 -0.967162  1.219199
1959-04-01  2.34  2778.801   5.1 -0.744763 -0.137503  0.967468
1959-07-01  2.74  2775.488   5.3  0.962570  0.473032  0.494297
1959-10-01  0.27  2785.204   5.6  0.053201 -0.514654 -0.859264
1960-01-01  2.31  2847.699   5.2 -1.523664  0.206165  0.476449

请注意，使用 `pivot` 功能其实相当于先通过 `set_index` 创建一个层次化索引，然后再调用 `unstack` 函数来处理数据。

In [107]:
unstacked = long_data.set_index(['date', 'item']).unstack(level='item')
unstacked.head()

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8  2.322029 -0.967162  1.219199
1959-04-01  2.34  2778.801   5.1 -0.744763 -0.137503  0.967468
1959-07-01  2.74  2775.488   5.3  0.962570  0.473032  0.494297
1959-10-01  0.27  2785.204   5.6  0.053201 -0.514654 -0.859264
1960-01-01  2.31  2847.699   5.2 -1.523664  0.206165  0.476449

### 将显示格式从“宽屏”切换为“长屏”

对于 DataFrame 来说，与`pivot`操作相反的是 `pandas.melt`。这个函数不是将一个列转换成多个列来生成新的 DataFrame，而是将多个列合并成一个列，从而生成一个比输入 DataFrame 更长的新 DataFrame。让我们来看一个例子：

In [108]:
df = pd.DataFrame({"key": ['foo', 'bar', 'baz'],
                   "A": [1, 2, 3],
                   "B": [4, 5, 6],
                   "C": [7, 8, 9]})
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


`key`列可以作为分组标识符，而其他列则包含数据值。在使用`pandas.melt`函数时，我们必须明确指出哪些列是分组标识符。在这里，我们以`key`列作为唯一的分组标识符：

In [110]:
melted = pd.melt(df, id_vars=['key'])
melted

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6
6,foo,C,7
7,bar,C,8
8,baz,C,9


通过使用`pivot`功能，我们可以将布局恢复到原来的状态:

In [111]:
reshaped = melted.pivot(index='key', columns='variable', values='value')
reshaped

variable,A,B,C
key,,,
bar,2,5,8
baz,3,6,9
foo,1,4,7


由于使用`pivot`函数后生成的数据索引是根据被用作行标签的列来创建的，因此我们可能需要使用`reset_index`函数将数据重新排列回列的形式:

In [112]:
reshaped.reset_index()

variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7


你也可以指定某些列作为值列来使用:

In [113]:
pd.melt(df, id_vars=['key'], value_vars=['A', 'B'])

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6


In [114]:
pd.melt(df, value_vars=['key', 'A', 'B'])

,variable,value
0,key,foo
1,key,bar
2,key,baz
3,A,1
4,A,2
5,A,3
6,B,4
7,B,5
8,B,6
